# ⚽ Simulateur CdM 2026 — Basé sur Modèle de prédiction MATCH
## Simulation complète : Groupes → Round of 32 → Finale
**FrenchTeam — Wild Code School — Juin 2026**

### 📋 Structure
1. Le modèle Match
2. Simuler les 12 groupes
3. Simuler les phases éliminatoires
4. Afficher le vainqueur prédit

## 📦 Étape 1 — Chargement du modèle hybride v3

> ⚠️ Cette cellule charge le modèle déjà entraîné (`models/modele_hybride.pkl`, voir `modele_hybride_v3.ipynb`).
> Pas de réentraînement ici — juste un chargement rapide du `.pkl`.
> Lance-la AVANT toutes les autres.


In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

DOSSIER_PROJET = r'C:\Users\Piwi\Documents\VS\WorldCup2026'
df = pd.read_csv(os.path.join(DOSSIER_PROJET, 'data', 'results.csv'))
df['date'] = pd.to_datetime(df['date'])
df = df.dropna(subset=['home_score', 'away_score'])
df_officiel = df[df['tournament'] != 'Friendly'].copy()

# ── Points FIFA officiels — Avril 2026
points_fifa = {
    'France': 1877.32, 'Spain': 1876.40, 'Argentina': 1874.81,
    'England': 1825.97, 'Portugal': 1763.83, 'Brazil': 1761.16,
    'Netherlands': 1757.87, 'Morocco': 1755.87, 'Belgium': 1734.71,
    'Germany': 1730.37, 'Croatia': 1717.07, 'Colombia': 1693.09,
    'Senegal': 1688.99, 'Mexico': 1681.03, 'United States': 1673.13,
    'Uruguay': 1673.07, 'Japan': 1660.43, 'Switzerland': 1649.40,
    'Ecuador': 1619.20, 'Turkey': 1614.55, 'Sweden': 1598.30,
    'Norway': 1590.12, 'Austria': 1578.44, 'South Korea': 1566.23,
    'Tunisia': 1542.18, 'Algeria': 1538.90, 'Ghana': 1521.44,
    'Egypt': 1518.77, 'Saudi Arabia': 1512.33, 'Iran': 1512.00,
    'Australia': 1508.66, 'Iraq': 1489.21, 'Czech Republic': 1488.00,
    'Scotland': 1487.55, 'Paraguay': 1481.33, 'Ivory Coast': 1479.88,
    'South Africa': 1421.34, 'Canada': 1418.77, 'Qatar': 1398.22,
    'Panama': 1392.11, 'Bosnia and Herzegovina': 1388.44,
    'Jordan': 1342.18, 'Uzbekistan': 1338.90, 'DR Congo': 1321.44,
    'New Zealand': 1298.77, 'Haiti': 1245.33, 'Curacao': 1198.22,
    'Cape Verde': 1450.00,
}

# ── Score qualité joueurs (Transfermarkt 2026 + présence de stars)
qualite_joueurs = {
    'France': 95, 'England': 90, 'Brazil': 92, 'Spain': 85,
    'Germany': 82, 'Argentina': 88, 'Portugal': 78, 'Netherlands': 75,
    'Belgium': 72, 'Japan': 65, 'Norway': 60, 'Colombia': 62,
    'Croatia': 58, 'Uruguay': 58, 'Morocco': 55, 'Senegal': 55,
    'Mexico': 55, 'United States': 52, 'Switzerland': 50, 'Turkey': 50,
    'Austria': 47, 'Canada': 45, 'South Korea': 45, 'Sweden': 42,
    'Ivory Coast': 40, 'Ecuador': 38, 'Egypt': 38, 'Algeria': 35,
    'Ghana': 35, 'Czech Republic': 35, 'Scotland': 33, 'Tunisia': 30,
    'Paraguay': 30, 'Bosnia and Herzegovina': 30, 'Saudi Arabia': 28,
    'Australia': 25, 'Iran': 25, 'DR Congo': 25, 'Iraq': 22,
    'Jordan': 20, 'South Africa': 20, 'Uzbekistan': 20,
    'Qatar': 18, 'Cape Verde': 15, 'Panama': 15,
    'New Zealand': 10, 'Haiti': 8, 'Curacao': 7,
}

# ── Coefficient expérience CdM
coefficient_experience = {
    'France': 1.00, 'Brazil': 1.00, 'Germany': 1.00, 'Spain': 1.00,
    'Argentina': 1.00, 'England': 1.00, 'Portugal': 1.00,
    'Netherlands': 1.00, 'Belgium': 1.00, 'Croatia': 1.00,
    'Uruguay': 1.00, 'Mexico': 1.00, 'United States': 1.00,
    'Canada': 0.95, 'Japan': 0.95, 'Morocco': 0.92, 'Senegal': 0.92,
    'South Korea': 0.92, 'Colombia': 0.90, 'Switzerland': 0.90,
    'Iran': 0.88, 'Saudi Arabia': 0.88, 'Ghana': 0.88, 'Ivory Coast': 0.88,
    'Ecuador': 0.85, 'Tunisia': 0.85, 'Algeria': 0.85, 'Austria': 0.85,
    'Sweden': 0.85, 'Turkey': 0.85, 'Egypt': 0.85, 'Paraguay': 0.85,
    'Czech Republic': 0.85, 'Norway': 0.50, 'Scotland': 0.52,
    'Australia': 0.48, 'Iraq': 0.70, 'Jordan': 0.60, 'Uzbekistan': 0.60,
    'Panama': 0.68, 'Curacao': 0.35, 'Cape Verde': 0.60,
    'New Zealand': 0.60, 'Haiti': 0.40, 'South Africa': 0.72,
    'Bosnia and Herzegovina': 0.75, 'DR Congo': 0.68, 'Qatar': 0.72,
}

# ── Zone de qualification
ZONE_QUALIF = {
    'France':'UEFA','Germany':'UEFA','Spain':'UEFA','England':'UEFA',
    'Portugal':'UEFA','Netherlands':'UEFA','Belgium':'UEFA','Croatia':'UEFA',
    'Switzerland':'UEFA','Austria':'UEFA','Turkey':'UEFA','Scotland':'UEFA',
    'Norway':'UEFA','Sweden':'UEFA','Bosnia and Herzegovina':'UEFA','Czech Republic':'UEFA',
    'Argentina':'CONMEBOL','Brazil':'CONMEBOL','Colombia':'CONMEBOL',
    'Uruguay':'CONMEBOL','Ecuador':'CONMEBOL','Paraguay':'CONMEBOL',
    'Morocco':'CAF','Senegal':'CAF','Egypt':'CAF','Ivory Coast':'CAF',
    'Algeria':'CAF','Tunisia':'CAF','Ghana':'CAF','South Africa':'CAF',
    'DR Congo':'CAF','Cape Verde':'CAF',
    'United States':'CONCACAF','Mexico':'CONCACAF','Canada':'CONCACAF',
    'Panama':'CONCACAF','Haiti':'CONCACAF','Curacao':'CONCACAF',
    'Japan':'AFC','South Korea':'AFC','Iran':'AFC','Saudi Arabia':'AFC',
    'Australia':'AFC','Uzbekistan':'AFC','Iraq':'AFC','Jordan':'AFC','Qatar':'AFC',
    'New Zealand':'OFC',
}
CREDIBILITE_ZONE = {
    'UEFA':1.00,'CONMEBOL':0.95,'CAF':0.75,'AFC':0.70,'CONCACAF':0.50,'OFC':0.40,
}

MOYENNE_FIFA    = np.mean(list(points_fifa.values()))
MOYENNE_QUALITE = np.mean(list(qualite_joueurs.values()))

def get_fifa(e):   return points_fifa.get(e, MOYENNE_FIFA)
def get_qual(e):   return qualite_joueurs.get(e, MOYENNE_QUALITE)
def get_coeff(e):  return coefficient_experience.get(e, 0.80)
def get_cred(e):   return CREDIBILITE_ZONE.get(ZONE_QUALIF.get(e, 'UEFA'), 0.75)

# ── Forme récente (identique au modèle hybride v3)
def calculer_forme(equipe, date, df_off, n=5):
    dom = df_off[(df_off['home_team']==equipe) & (df_off['date']<date)].tail(n)
    ext = df_off[(df_off['away_team']==equipe) & (df_off['date']<date)].tail(n)
    matchs = pd.concat([dom, ext]).sort_values('date').tail(n)
    if len(matchs) == 0: return 0.5
    pts = pw = 0
    for _, m in matchs.iterrows():
        if m['home_team'] == equipe:
            adv = m['away_team']; v = m['home_score'] > m['away_score']; nu = m['home_score'] == m['away_score']
        else:
            adv = m['home_team']; v = m['away_score'] > m['home_score']; nu = m['home_score'] == m['away_score']
        p = get_fifa(adv) / 1500
        if v: pts += 3 * p
        elif nu: pts += 1 * p
        pw += 3 * p
    fb = pts / pw if pw > 0 else 0.5
    coeff_final = get_coeff(equipe) * get_cred(equipe)
    BONUS = {'Argentina':0.10,'France':0.08,'Brazil':0.10,'Germany':0.06,'Spain':0.05}
    forme = min(0.5 + (fb - 0.5) * coeff_final + BONUS.get(equipe, 0), 1.0)
    PLANCHER = {'Brazil':0.75,'Germany':0.72,'Spain':0.80,'France':0.82,
                'Argentina':0.85,'England':0.78,'Portugal':0.75}
    return max(forme, PLANCHER.get(equipe, 0.0))

# ── Score de Force composite — coeur du modèle hybride v3
# FORCE = 0.50 x qualite_squad + 0.35 x classement_FIFA + 0.15 x forme_recente
FIFA_MIN,  FIFA_MAX  = 1198,  1877
QUAL_MIN,  QUAL_MAX  = 7,     95
FORME_MIN, FORME_MAX = 0.25,  0.97
POIDS_FORCE = {'qualite': 0.50, 'fifa': 0.35, 'forme': 0.15}

def normaliser(val, vmin, vmax):
    return max(0.0, min(1.0, (val - vmin) / (vmax - vmin)))

def score_force(equipe, date=None, df_off=None):
    n_qual  = normaliser(get_qual(equipe),  QUAL_MIN,  QUAL_MAX)
    n_fifa  = normaliser(get_fifa(equipe),  FIFA_MIN,  FIFA_MAX)
    n_forme = normaliser(
        calculer_forme(equipe, date, df_off) if date is not None else 0.5,
        FORME_MIN, FORME_MAX
    )
    return (POIDS_FORCE['qualite'] * n_qual +
            POIDS_FORCE['fifa']    * n_fifa +
            POIDS_FORCE['forme']   * n_forme)

# ── Chargement du modèle hybride v3 déjà entraîné (notebooks/modele_hybride_v3.ipynb)
# Pas de réentraînement ici : on réutilise directement le .pkl sauvegardé.
modele = joblib.load(os.path.join(DOSSIER_PROJET, 'models', 'modele_hybride.pkl'))
enc    = joblib.load(os.path.join(DOSSIER_PROJET, 'models', 'encodeur_cible_hybride.pkl'))

CLASSES = list(enc.classes_)
IA, IB, IN = CLASSES.index('A_gagne'), CLASSES.index('B_gagne'), CLASSES.index('Nul')
HOTES   = ['United States', 'Canada', 'Mexico']
DATE_T  = pd.Timestamp('2026-06-11')
ALPHA   = 0.5  # compression des probas vers 1/3 (probas plus réalistes face aux bookmakers)

def corriger_coherence(p_a, p_n, p_b, force_a, force_b):
    """Corrige si le modèle contredit le score de force."""
    ecart = force_a - force_b
    if abs(ecart) < 0.05: return p_a, p_n, p_b
    a_gagne = ecart > 0
    if not ((a_gagne and p_b > p_a) or (not a_gagne and p_a > p_b)):
        return p_a, p_n, p_b
    force_corr = min(0.35, abs(ecart) * 0.5)
    if a_gagne:
        pa2=p_a+force_corr; pb2=max(0.08,p_b-force_corr*0.85); pn2=max(0.10,p_n-force_corr*0.15)
    else:
        pb2=p_b+force_corr; pa2=max(0.08,p_a-force_corr*0.85); pn2=max(0.10,p_n-force_corr*0.15)
    t=pa2+pn2+pb2
    return pa2/t, pn2/t, pb2/t

# ── Fonction principale de prédiction — utilisée par tout le reste du notebook
def probas_match(eq_a, eq_b):
    """Retourne (p_a, p_nul, p_b) avec le modèle hybride v3 (score de force + XGBoost)."""
    fa = score_force(eq_a, DATE_T, df_officiel)
    fb = score_force(eq_b, DATE_T, df_officiel)
    h  = 0.5 if eq_a in HOTES else (-0.5 if eq_b in HOTES else 0.0)
    ef_ab = fa - fb
    fm    = (fa + fb) / 2

    feat_ab = pd.DataFrame([{
        'ecart_force':       ef_ab,
        'ecart_force_carre': ef_ab**2 * np.sign(ef_ab),
        'abs_ecart_force':   abs(ef_ab),
        'force_moy':         fm,
        'avantage_hote':     h,
    }])
    feat_ba = pd.DataFrame([{
        'ecart_force':       -ef_ab,
        'ecart_force_carre': (-ef_ab)**2 * np.sign(-ef_ab),
        'abs_ecart_force':   abs(ef_ab),
        'force_moy':         fm,
        'avantage_hote':     -h,
    }])

    pab = modele.predict_proba(feat_ab)[0]
    pba = modele.predict_proba(feat_ba)[0]

    pa = (pab[IA]+pba[IB])/2; pb = (pab[IB]+pba[IA])/2; pn = (pab[IN]+pba[IN])/2
    t = pa+pb+pn; pa/=t; pb/=t; pn/=t

    pa = 1/3 + ALPHA*(pa - 1/3)
    pn = 1/3 + ALPHA*(pn - 1/3)
    pb = 1/3 + ALPHA*(pb - 1/3)
    t = pa+pn+pb; pa/=t; pn/=t; pb/=t

    return corriger_coherence(pa, pn, pb, fa, fb)

def vainqueur_match(eq_a, eq_b):
    """Retourne le vainqueur (pas de nul — pour phases éliminatoires)."""
    pa, _, pb = probas_match(eq_a, eq_b)
    return eq_a if pa >= pb else eq_b

print('✅ Modèle hybride v3 chargé (models/modele_hybride.pkl) — prêt pour la simulation !')


✅ Modèle hybride v3 chargé (models/modele_hybride.pkl) — prêt pour la simulation !


## 🏟️ Étape 2 — Groupes CdM 2026

Les 12 groupes officiels du tirage au sort de décembre 2024.

### Format
- 4 équipes par groupe
- 6 matchs par groupe (chacun joue contre les 3 autres)
- Les 2 premiers + les 8 meilleurs 3es se qualifient
- **32 qualifiés au total** pour le Round of 32

In [2]:
# ── 12 Groupes officiels CdM 2026
GROUPES = {
    'A': ['Mexico',        'South Africa',  'South Korea',           'Czech Republic'],
    'B': ['Canada',        'Bosnia and Herzegovina', 'Qatar',         'Switzerland'],
    'C': ['Brazil',        'Morocco',       'Haiti',                 'Scotland'],
    'D': ['United States', 'Paraguay',      'Australia',             'Turkey'],
    'E': ['Germany',       'Curacao',       'Ivory Coast',           'Ecuador'],
    'F': ['Netherlands',   'Japan',         'Sweden',                'Tunisia'],
    'G': ['Belgium',       'Egypt',         'Iran',                  'New Zealand'],
    'H': ['Spain',         'Cape Verde',    'Saudi Arabia',          'Uruguay'],
    'I': ['France',        'Senegal',       'Iraq',                  'Norway'],
    'J': ['Argentina',     'Algeria',       'Austria',               'Jordan'],
    'K': ['Portugal',      'DR Congo',      'Uzbekistan',            'Colombia'],
    'L': ['England',       'Croatia',       'Ghana',                 'Panama'],
}

print('Groupes CdM 2026 :')
for g, teams in GROUPES.items():
    print(f'  Groupe {g}: {" | ".join(teams)}')

Groupes CdM 2026 :
  Groupe A: Mexico | South Africa | South Korea | Czech Republic
  Groupe B: Canada | Bosnia and Herzegovina | Qatar | Switzerland
  Groupe C: Brazil | Morocco | Haiti | Scotland
  Groupe D: United States | Paraguay | Australia | Turkey
  Groupe E: Germany | Curacao | Ivory Coast | Ecuador
  Groupe F: Netherlands | Japan | Sweden | Tunisia
  Groupe G: Belgium | Egypt | Iran | New Zealand
  Groupe H: Spain | Cape Verde | Saudi Arabia | Uruguay
  Groupe I: France | Senegal | Iraq | Norway
  Groupe J: Argentina | Algeria | Austria | Jordan
  Groupe K: Portugal | DR Congo | Uzbekistan | Colombia
  Groupe L: England | Croatia | Ghana | Panama


## 📊 Étape 3 — Simulation des groupes

Chaque match de groupe est simulé via le modèle hybride v3.
Les points sont calculés (V=3, N=1, D=0), le classement établi.

In [3]:
def simuler_groupe(nom_groupe, equipes):
    """
    Simule un groupe de 4 équipes.
    Retourne le classement et les résultats de chaque match.
    Critères départage : points → diff buts → buts marqués (estimé via probas)
    """
    # Initialiser le tableau
    tableau = {eq: {'pts':0, 'j':0, 'g':0, 'n':0, 'p':0, 'bp':0, 'bc':0} for eq in equipes}
    resultats = []

    # Simuler les 6 matchs
    from itertools import combinations
    for eq_a, eq_b in combinations(equipes, 2):
        pa, pn, pb = probas_match(eq_a, eq_b)

        # Déterminer le résultat le plus probable
        if pa >= pb and pa >= pn:
            res = 'A'
            tableau[eq_a]['pts'] += 3; tableau[eq_a]['g'] += 1
            tableau[eq_b]['p'] += 1
            score = '2-0'
        elif pb > pa and pb >= pn:
            res = 'B'
            tableau[eq_b]['pts'] += 3; tableau[eq_b]['g'] += 1
            tableau[eq_a]['p'] += 1
            score = '0-2'
        else:
            res = 'N'
            tableau[eq_a]['pts'] += 1; tableau[eq_a]['n'] += 1
            tableau[eq_b]['pts'] += 1; tableau[eq_b]['n'] += 1
            score = '1-1'

        # Mise à jour matchs joués
        tableau[eq_a]['j'] += 1; tableau[eq_b]['j'] += 1

        resultats.append({
            'match': f'{eq_a} vs {eq_b}',
            'score': score,
            'pa': pa, 'pn': pn, 'pb': pb,
            'resultat': res
        })

    # Classement par points
    classement = sorted(equipes, key=lambda e: tableau[e]['pts'], reverse=True)

    return classement, tableau, resultats


def afficher_groupe(nom, classement, tableau, resultats):
    """Affiche le groupe avec le tableau et les résultats."""
    print(f'\n{'='*55}')
    print(f'  GROUPE {nom}')
    print(f'{'='*55}')

    # Tableau du groupe
    print(f'  {"Équipe":<25} {"Pts":>4} {"J":>2} {"G":>2} {"N":>2} {"P":>2}')
    print(f'  {"-"*44}')
    for i, eq in enumerate(classement):
        t = tableau[eq]
        qualif = '✅' if i < 2 else '  '
        print(f'  {qualif} {eq:<23} {t["pts"]:>4} {t["j"]:>2} {t["g"]:>2} {t["n"]:>2} {t["p"]:>2}')

    # Résultats des matchs
    print(f'\n  Matchs :')
    for r in resultats:
        icone = '→' if r['resultat'] == 'A' else ('←' if r['resultat'] == 'B' else '=')
        a_name = r['match'].split(' vs ')[0]
        b_name = r['match'].split(' vs ')[1]
        print(f'    {a_name:<22} {icone}  {b_name:<22} '
              f'({r["pa"]:.0%} / {r["pn"]:.0%} / {r["pb"]:.0%})')


# ── Simuler tous les groupes
print('Simulation des 12 groupes en cours...\n')
resultats_groupes = {}

for nom, equipes in GROUPES.items():
    classement, tableau, resultats = simuler_groupe(nom, equipes)
    resultats_groupes[nom] = {
        'classement': classement,
        'tableau': tableau,
        'resultats': resultats
    }
    afficher_groupe(nom, classement, tableau, resultats)

print('\n✅ Groupes simulés !')

Simulation des 12 groupes en cours...


  GROUPE A
  Équipe                     Pts  J  G  N  P
  --------------------------------------------
  ✅ Mexico                     9  3  3  0  0
  ✅ South Korea                6  3  2  0  1
     South Africa               3  3  1  0  2
     Czech Republic             0  3  0  0  3

  Matchs :
    Mexico                 →  South Africa           (48% / 31% / 21%)
    Mexico                 →  South Korea            (46% / 29% / 24%)
    Mexico                 →  Czech Republic         (48% / 29% / 23%)
    South Africa           ←  South Korea            (32% / 32% / 37%)
    South Africa           →  Czech Republic         (37% / 29% / 35%)
    South Korea            →  Czech Republic         (36% / 36% / 28%)

  GROUPE B
  Équipe                     Pts  J  G  N  P
  --------------------------------------------
  ✅ Switzerland                9  3  3  0  0
  ✅ Canada                     6  3  2  0  1
     Bosnia and Herzegovina     3  3  1  0 

## 🏆 Étape 4 — Qualifiés et tirage Round of 32

### Format CdM 2026
- **32 qualifiés** : 1er et 2e de chaque groupe (24) + 8 meilleurs 3es
- Round of 32 → Round of 16 → Quarts → Demis → Finale

In [4]:
# ── Récupérer les qualifiés
premiers = {}   # 1er de chaque groupe
deuxiemes = {}  # 2e de chaque groupe
troisiemes = [] # 3es pour les wild cards

for nom, data in resultats_groupes.items():
    cl = data['classement']
    tb = data['tableau']
    premiers[nom]   = cl[0]
    deuxiemes[nom]  = cl[1]
    troisiemes.append((cl[2], tb[cl[2]]['pts'], nom))

# Trier les 3es par points → les 8 meilleurs se qualifient
troisiemes_sorted = sorted(troisiemes, key=lambda x: x[1], reverse=True)
qualifies_3es = [t[0] for t in troisiemes_sorted[:8]]

print('=' * 55)
print('QUALIFIÉS POUR LE ROUND OF 32')
print('=' * 55)

print('\n1ers de groupe (12) :')
for g, eq in premiers.items():
    print(f'  Groupe {g} : {eq}')

print('\n2es de groupe (12) :')
for g, eq in deuxiemes.items():
    print(f'  Groupe {g} : {eq}')

print('\n8 meilleurs 3es :')
for eq, pts, g in troisiemes_sorted[:8]:
    print(f'  Groupe {g} : {eq} ({pts} pts)')

print(f'\nTotal qualifiés : {len(premiers)+len(deuxiemes)+len(qualifies_3es)}')

QUALIFIÉS POUR LE ROUND OF 32

1ers de groupe (12) :
  Groupe A : Mexico
  Groupe B : Switzerland
  Groupe C : Brazil
  Groupe D : United States
  Groupe E : Germany
  Groupe F : Netherlands
  Groupe G : Belgium
  Groupe H : Spain
  Groupe I : France
  Groupe J : Argentina
  Groupe K : Portugal
  Groupe L : England

2es de groupe (12) :
  Groupe A : South Korea
  Groupe B : Canada
  Groupe C : Morocco
  Groupe D : Turkey
  Groupe E : Ecuador
  Groupe F : Japan
  Groupe G : Iran
  Groupe H : Uruguay
  Groupe I : Senegal
  Groupe J : Austria
  Groupe K : Colombia
  Groupe L : Croatia

8 meilleurs 3es :
  Groupe I : Norway (4 pts)
  Groupe A : South Africa (3 pts)
  Groupe B : Bosnia and Herzegovina (3 pts)
  Groupe C : Scotland (3 pts)
  Groupe D : Paraguay (3 pts)
  Groupe E : Ivory Coast (3 pts)
  Groupe F : Sweden (3 pts)
  Groupe G : Egypt (3 pts)

Total qualifiés : 32


## ⚔️ Étape 5 — Phases éliminatoires

En phases éliminatoires, pas de nul possible — le favori (probabilité la plus haute) gagne.

In [5]:
def simuler_phase(matchs, nom_phase):
    print(f'\n{"-"*55}')
    print(f'  {nom_phase}')
    print(f'{"-"*55}')
    vainqueurs = []
    for eq_a, eq_b in matchs:
        pa, pn, pb = probas_match(eq_a, eq_b)
        pa_el = pa + pn * pa / (pa + pb)
        pb_el = pb + pn * pb / (pa + pb)
        gagnant = eq_a if pa_el >= pb_el else eq_b
        vainqueurs.append(gagnant)
        print(f'  {eq_a:<25} {pa_el:.0%}  vs  {pb_el:.0%}  {eq_b}')
        print(f'    → {gagnant} se qualifie')
    return vainqueurs


# ══════════════════════════════════════════════════════════
# ✅ TABLE OFFICIELLE FIFA — 45 combinaisons principales sur 495
# Source : Annexe C règlement FIFA + Wikipedia EN knockout stage
# Format : frozenset(groupes_3es) → {slot_1er: groupe_du_3e}
# ══════════════════════════════════════════════════════════
TABLE_3ES = {
    frozenset(['E','F','G','H','I','J','K','L']): {'1A':'E','1B':'J','1D':'I','1E':'F','1G':'H','1I':'G','1K':'L','1L':'K'},
    frozenset(['D','F','G','H','I','J','K','L']): {'1A':'H','1B':'G','1D':'I','1E':'D','1G':'J','1I':'F','1K':'L','1L':'K'},
    frozenset(['D','E','G','H','I','J','K','L']): {'1A':'E','1B':'J','1D':'I','1E':'D','1G':'H','1I':'G','1K':'L','1L':'K'},
    frozenset(['D','E','F','H','I','J','K','L']): {'1A':'E','1B':'J','1D':'I','1E':'D','1G':'H','1I':'F','1K':'L','1L':'K'},
    frozenset(['D','E','F','G','I','J','K','L']): {'1A':'E','1B':'G','1D':'I','1E':'D','1G':'J','1I':'F','1K':'L','1L':'K'},
    frozenset(['D','E','F','G','H','J','K','L']): {'1A':'E','1B':'G','1D':'J','1E':'D','1G':'H','1I':'F','1K':'L','1L':'K'},
    frozenset(['D','E','F','G','H','I','K','L']): {'1A':'E','1B':'G','1D':'I','1E':'D','1G':'H','1I':'F','1K':'L','1L':'K'},
    frozenset(['D','E','F','G','H','I','J','L']): {'1A':'E','1B':'G','1D':'J','1E':'D','1G':'H','1I':'F','1K':'L','1L':'I'},
    frozenset(['D','E','F','G','H','I','J','K']): {'1A':'E','1B':'G','1D':'J','1E':'D','1G':'H','1I':'F','1K':'I','1L':'K'},
    frozenset(['C','E','F','G','H','I','J','K']): {'1A':'E','1B':'G','1D':'J','1E':'C','1G':'H','1I':'F','1K':'I','1L':'K'},
    frozenset(['C','D','E','F','G','H','I','J']): {'1A':'C','1B':'G','1D':'J','1E':'D','1G':'H','1I':'F','1K':'E','1L':'I'},
    frozenset(['C','D','E','F','G','H','I','K']): {'1A':'C','1B':'G','1D':'E','1E':'D','1G':'H','1I':'F','1K':'I','1L':'K'},
    frozenset(['C','D','E','F','G','H','I','L']): {'1A':'C','1B':'G','1D':'E','1E':'D','1G':'H','1I':'F','1K':'L','1L':'I'},
    frozenset(['C','D','E','F','G','H','J','K']): {'1A':'C','1B':'G','1D':'J','1E':'D','1G':'H','1I':'F','1K':'E','1L':'K'},
    frozenset(['C','D','E','F','G','H','J','L']): {'1A':'C','1B':'G','1D':'J','1E':'D','1G':'H','1I':'F','1K':'L','1L':'E'},
    frozenset(['C','D','E','F','G','H','K','L']): {'1A':'C','1B':'G','1D':'E','1E':'D','1G':'H','1I':'F','1K':'L','1L':'K'},
    frozenset(['C','D','E','F','G','I','J','K']): {'1A':'C','1B':'G','1D':'E','1E':'D','1G':'J','1I':'F','1K':'I','1L':'K'},
    frozenset(['C','D','E','F','G','I','J','L']): {'1A':'C','1B':'G','1D':'E','1E':'D','1G':'J','1I':'F','1K':'L','1L':'I'},
    frozenset(['C','D','E','F','G','I','K','L']): {'1A':'C','1B':'G','1D':'E','1E':'D','1G':'I','1I':'F','1K':'L','1L':'K'},
    frozenset(['C','D','E','F','G','J','K','L']): {'1A':'C','1B':'G','1D':'E','1E':'D','1G':'J','1I':'F','1K':'L','1L':'K'},
    frozenset(['C','D','E','F','H','I','J','K']): {'1A':'C','1B':'J','1D':'E','1E':'D','1G':'H','1I':'F','1K':'I','1L':'K'},
    frozenset(['C','D','E','F','H','I','J','L']): {'1A':'C','1B':'J','1D':'E','1E':'D','1G':'H','1I':'F','1K':'L','1L':'I'},
    frozenset(['C','D','E','F','H','I','K','L']): {'1A':'C','1B':'E','1D':'I','1E':'D','1G':'H','1I':'F','1K':'L','1L':'K'},
    frozenset(['C','D','E','F','H','J','K','L']): {'1A':'C','1B':'J','1D':'E','1E':'D','1G':'H','1I':'F','1K':'L','1L':'K'},
    frozenset(['C','D','E','F','I','J','K','L']): {'1A':'C','1B':'J','1D':'E','1E':'D','1G':'I','1I':'F','1K':'L','1L':'K'},
}

SLOT_VERS_MATCH = {
    '1E':'74','1I':'77','1A':'79','1L':'80',
    '1D':'81','1G':'82','1B':'85','1K':'87'
}

def assigner_3es_officiel(troisiemes_sorted):
    """
    ✅ Attribution des 3es selon la table officielle FIFA.
    Utilise la combinaison exacte si disponible, sinon fallback.
    """
    groupes_3es = frozenset(grp for _, _, grp in troisiemes_sorted[:8])
    equipe_par_groupe = {grp: eq for eq, _, grp in troisiemes_sorted[:8]}

    if groupes_3es in TABLE_3ES:
        assignation = TABLE_3ES[groupes_3es]
        print(f'  ✅ Combinaison FIFA exacte : groupes {sorted(groupes_3es)}')
        slots = {}
        for slot_nom, grp in assignation.items():
            match_num = SLOT_VERS_MATCH[slot_nom]
            slots[match_num] = equipe_par_groupe.get(grp, list(equipe_par_groupe.values())[0])
        return slots
    else:
        print(f'  ⚠️ Combinaison non trouvée → fallback')
        SLOTS_FB = [
            ('74',['A','B','C','D','F']),('77',['C','D','F','G','H']),
            ('79',['C','E','F','H','I']),('80',['E','H','I','J','K']),
            ('81',['B','E','F','I','J']),('82',['A','E','H','I','J']),
            ('85',['E','F','G','I','J']),('87',['D','E','I','J','L']),
        ]
        dispo = list(troisiemes_sorted[:8])
        slots = {}
        for slot, grps in SLOTS_FB:
            for j,(eq,_,grp) in enumerate(dispo):
                if grp in grps: slots[slot]=eq; dispo.pop(j); break
            else:
                if dispo: slots[slot]=dispo.pop(0)[0]
        return slots


# ✅ FORMAT OFFICIEL FIFA 2026 — Round of 32
# M73:2A/2B  M74:1E/3e  M75:1F/2C  M76:1C/2F
# M77:1I/3e  M78:2E/2I  M79:1A/3e  M80:1L/3e
# M81:1D/3e  M82:1G/3e  M83:2K/2L  M84:1H/2J
# M85:1B/3e  M86:1J/2H  M87:1K/3e  M88:2D/2G
slots_3es = assigner_3es_officiel(troisiemes_sorted)

print('\nAssignation 3es — table officielle FIFA :')
labels = {'74':'1E','77':'1I','79':'1A','80':'1L','81':'1D','82':'1G','85':'1B','87':'1K'}
for s, eq in sorted(slots_3es.items()):
    print(f'  M{s} : {labels.get(s,"?")} vs {eq}')

r32_matchs = [
    (deuxiemes['A'],  deuxiemes['B']),           # M73 : 2A vs 2B
    (premiers['E'],   slots_3es.get('74')),       # M74 : 1E vs 3e(A/B/C/D/F)
    (premiers['F'],   deuxiemes['C']),            # M75 : 1F vs 2C
    (premiers['C'],   deuxiemes['F']),            # M76 : 1C vs 2F
    (premiers['I'],   slots_3es.get('77')),       # M77 : 1I vs 3e(C/D/F/G/H)
    (deuxiemes['E'],  deuxiemes['I']),            # M78 : 2E vs 2I
    (premiers['A'],   slots_3es.get('79')),       # M79 : 1A vs 3e(C/E/F/H/I)
    (premiers['L'],   slots_3es.get('80')),       # M80 : 1L vs 3e(E/H/I/J/K)
    (premiers['D'],   slots_3es.get('81')),       # M81 : 1D vs 3e(B/E/F/I/J)
    (premiers['G'],   slots_3es.get('82')),       # M82 : 1G vs 3e(A/E/H/I/J)
    (deuxiemes['K'],  deuxiemes['L']),            # M83 : 2K vs 2L
    (premiers['H'],   deuxiemes['J']),            # M84 : 1H vs 2J
    (premiers['B'],   slots_3es.get('85')),       # M85 : 1B vs 3e(E/F/G/I/J)
    (premiers['J'],   deuxiemes['H']),            # M86 : 1J vs 2H
    (premiers['K'],   slots_3es.get('87')),       # M87 : 1K vs 3e(D/E/I/J/L)
    (deuxiemes['D'],  deuxiemes['G']),            # M88 : 2D vs 2G
]

toutes_eq = [eq for p in r32_matchs for eq in p]
assert len(toutes_eq) == len(set(toutes_eq)), '❌ DOUBLON !'
print(f'\n✅ R32 FIFA officiel : {len(r32_matchs)} matchs, {len(toutes_eq)} équipes uniques')

r16_qualifies = simuler_phase(r32_matchs, 'ROUND OF 32 — Format officiel FIFA 2026')
print(f'\n✅ {len(r16_qualifies)} équipes qualifiées pour le Round of 16')


  ⚠️ Combinaison non trouvée → fallback

Assignation 3es — table officielle FIFA :
  M74 : 1E vs South Africa
  M77 : 1I vs Scotland
  M79 : 1A vs Norway
  M80 : 1L vs Ivory Coast
  M81 : 1D vs Bosnia and Herzegovina
  M82 : 1G vs Paraguay
  M85 : 1B vs Sweden
  M87 : 1K vs Egypt

✅ R32 FIFA officiel : 16 matchs, 32 équipes uniques

-------------------------------------------------------
  ROUND OF 32 — Format officiel FIFA 2026
-------------------------------------------------------
  South Korea               53%  vs  47%  Canada
    → South Korea se qualifie
  Germany                   74%  vs  26%  South Africa
    → Germany se qualifie
  Netherlands               61%  vs  39%  Morocco
    → Netherlands se qualifie
  Brazil                    62%  vs  38%  Japan
    → Brazil se qualifie
  France                    72%  vs  28%  Scotland
    → France se qualifie
  Ecuador                   34%  vs  66%  Senegal
    → Senegal se qualifie
  Mexico                    58%  vs  42%  Norw

In [6]:
# ══════════════════════════════════════════════════════════
# ✅ BRACKET OFFICIEL FIFA 2026 — R16 → Finale
# Source : Wikipedia FR — Phase finale CdM 2026
#
# Ordre des 16 vainqueurs dans r16_qualifies :
# idx 0  = V(M73) 2A/2B        idx 1  = V(M74) 1E/3e
# idx 2  = V(M75) 1F/2C        idx 3  = V(M76) 1C/2F
# idx 4  = V(M77) 1I/3e        idx 5  = V(M78) 2E/2I
# idx 6  = V(M79) 1A/3e        idx 7  = V(M80) 1L/3e
# idx 8  = V(M81) 1D/3e        idx 9  = V(M82) 1G/3e
# idx 10 = V(M83) 2K/2L        idx 11 = V(M84) 1H/2J
# idx 12 = V(M85) 1B/3e        idx 13 = V(M86) 1J/2H
# idx 14 = V(M87) 1K/3e        idx 15 = V(M88) 2D/2G
# ══════════════════════════════════════════════════════════

v = r16_qualifies  # raccourci

# ── Round of 16
# CÔTÉ DALLAS (AT&T Stadium, Arlington)
r16_dallas = [
    (v[1],  v[4]),   # R16_A : V(M74) vs V(M77)  — 1E/3e vs 1I/3e
    (v[0],  v[2]),   # R16_B : V(M73) vs V(M75)  — 2A/2B vs 1F/2C
    (v[10], v[11]),  # R16_C : V(M83) vs V(M84)  — 2K/2L vs 1H/2J
    (v[8],  v[9]),   # R16_D : V(M81) vs V(M82)  — 1D/3e vs 1G/3e
]
# CÔTÉ ATLANTA (Mercedes-Benz Stadium)
r16_atlanta = [
    (v[3],  v[5]),   # R16_E : V(M76) vs V(M78)  — 1C/2F vs 2E/2I
    (v[6],  v[7]),   # R16_F : V(M79) vs V(M80)  — 1A/3e vs 1L/3e
    (v[13], v[15]),  # R16_G : V(M86) vs V(M88)  — 1J/2H vs 2D/2G
    (v[12], v[14]),  # R16_H : V(M85) vs V(M87)  — 1B/3e vs 1K/3e
]

qf_dallas  = simuler_phase(r16_dallas,  'ROUND OF 16 — Côté Dallas  (4 matchs)')
qf_atlanta = simuler_phase(r16_atlanta, 'ROUND OF 16 — Côté Atlanta (4 matchs)')

# ── Quarts de finale
qf_matchs_dallas  = [(qf_dallas[0],  qf_dallas[1]),   # QF1 : R16_A vs R16_B
                     (qf_dallas[2],  qf_dallas[3])]   # QF2 : R16_C vs R16_D
qf_matchs_atlanta = [(qf_atlanta[0], qf_atlanta[1]),  # QF3 : R16_E vs R16_F
                     (qf_atlanta[2], qf_atlanta[3])]  # QF4 : R16_G vs R16_H

sf_dallas  = simuler_phase(qf_matchs_dallas,  'QUARTS DE FINALE — Dallas  (2 matchs)')
sf_atlanta = simuler_phase(qf_matchs_atlanta, 'QUARTS DE FINALE — Atlanta (2 matchs)')

# ── Demi-finales
demis = [
    (sf_dallas[0],  sf_dallas[1]),   # Demi Dallas  : QF1 vs QF2
    (sf_atlanta[0], sf_atlanta[1]),  # Demi Atlanta : QF3 vs QF4
]
finalistes = simuler_phase(demis, 'DEMI-FINALES (2 matchs)')

# ── Finale — MetLife Stadium, New Jersey (19 juillet 2026)
print(f'\n{"="*58}')
print(f'  FINALE — MetLife Stadium, New Jersey — 19 juillet 2026')
print(f'{"="*58}')
pa, pn, pb = probas_match(finalistes[0], finalistes[1])
pa_el = pa + pn * pa / (pa + pb)
pb_el = pb + pn * pb / (pa + pb)
champion  = finalistes[0] if pa_el >= pb_el else finalistes[1]
finaliste = finalistes[1] if champion == finalistes[0] else finalistes[0]
print(f'\n  {finalistes[0]:<25} {pa_el:.0%}  vs  {pb_el:.0%}  {finalistes[1]}')
print(f'\n  🏆 CHAMPION DU MONDE 2026 : {champion.upper()} 🏆')
print(f'  🥈 Finaliste               : {finaliste}')
print(f'\n  Demi Dallas  : {sf_dallas[0]}  vs  {sf_dallas[1]}')
print(f'  Demi Atlanta : {sf_atlanta[0]}  vs  {sf_atlanta[1]}')



-------------------------------------------------------
  ROUND OF 16 — Côté Dallas  (4 matchs)
-------------------------------------------------------
  Germany                   44%  vs  56%  France
    → France se qualifie
  South Korea               32%  vs  68%  Netherlands
    → Netherlands se qualifie
  Croatia                   40%  vs  60%  Spain
    → Spain se qualifie
  United States             34%  vs  66%  Belgium
    → Belgium se qualifie

-------------------------------------------------------
  ROUND OF 16 — Côté Atlanta (4 matchs)
-------------------------------------------------------
  Brazil                    63%  vs  37%  Senegal
    → Brazil se qualifie
  Mexico                    35%  vs  65%  England
    → England se qualifie
  Argentina                 67%  vs  33%  Turkey
    → Argentina se qualifie
  Switzerland               36%  vs  64%  Portugal
    → Portugal se qualifie

-------------------------------------------------------
  QUARTS DE FINALE — Dall

## 📊 Étape 6 — Résumé complet du tournoi

## 🎲 Étape 6 — Simulation Monte Carlo (1000 tournois)

Au lieu de toujours prendre le favori, on tire au sort selon les vraies probabilités.

- **Mode déterministe** (étapes précédentes) : toujours le même champion
- **Mode Monte Carlo** (cette étape) : simule 1000 tournois aléatoires

> ⚠️ Le résultat sera **différent à chaque execution** — c'est normal !
> Pour un résultat reproductible, fixe `random.seed(42)` à la place de `random.seed(None)`.

In [7]:
# ══════════════════════════════════════════════════════════
# SIMULATION MONTE CARLO
# Précalcul des probabilités en cache
# → réentraînement ~10 min, puis ~30 sec pour 5000 simulations
# ══════════════════════════════════════════════════════════

import random
from collections import Counter
from itertools import combinations
import time

N_SIMULATIONS = 5000
random.seed(None)

# Précalcul cache
toutes_equipes = list(set(eq for g in GROUPES.values() for eq in g))
CACHE = {}
t0 = time.time()
total = len(toutes_equipes) * (len(toutes_equipes) - 1)
print(f'Précalcul {total} paires...')
done = 0
for a in toutes_equipes:
    for b in toutes_equipes:
        if a != b:
            CACHE[(a,b)] = probas_match(a,b)
            done += 1
    if done % 100 == 0 and done > 0:
        print(f'  {done}/{total}...')
print(f'✅ Cache en {time.time()-t0:.0f}s')

def mg(a,b):
    pa,pn,pb = CACHE[(a,b)]
    r = random.random()
    return 'A' if r<pa else ('N' if r<pa+pn else 'B')

def me(a,b):
    pa,pn,pb = CACHE[(a,b)]
    pa_el = pa + pn*pa/(pa+pb)
    return a if random.random()<pa_el else b

def gmc(eq):
    pts={e:0 for e in eq}
    for a,b in combinations(eq,2):
        r=mg(a,b)
        if r=='A': pts[a]+=3
        elif r=='B': pts[b]+=3
        else: pts[a]+=1; pts[b]+=1
    return sorted(eq,key=lambda e:-pts[e])

def tmc():
    p,d,t={},{},[]
    for n,eq in GROUPES.items():
        cl=gmc(eq); p[n]=cl[0]; d[n]=cl[1]; t.append((cl[2],0,n))
    # ✅ Table officielle FIFA pour attribution des 3es
    grps=frozenset(g for _,_,g in t[:8]); epg={g:e for e,_,g in t[:8]}
    SM={'1E':'74','1I':'77','1A':'79','1L':'80','1D':'81','1G':'82','1B':'85','1K':'87'}
    if grps in TABLE_3ES:
        s3={SM[k]:epg.get(v,list(epg.values())[0]) for k,v in TABLE_3ES[grps].items()}
    else:
        SFB=[['A','B','C','D','F'],['C','D','F','G','H'],['C','E','F','H','I'],
             ['E','H','I','J','K'],['B','E','F','I','J'],['A','E','H','I','J'],
             ['E','F','G','I','J'],['D','E','I','J','L']]
        dispo=list(t[:8]); s3={}; ks=['74','77','79','80','81','82','85','87']
        for ki,gr in enumerate(SFB):
            for ji,(eq,_,grp) in enumerate(dispo):
                if grp in gr: s3[ks[ki]]=eq; dispo.pop(ji); break
            else:
                if dispo: s3[ks[ki]]=dispo.pop(0)[0]
    # ✅ Format FIFA officiel
    SLOTS=[['A','B','C','D','F'],['C','D','F','G','H'],
           ['C','E','F','H','I'],['E','H','I','J','K'],
           ['B','E','F','I','J'],['A','E','H','I','J'],
           ['E','F','G','I','J'],['D','E','I','J','L']]
    dispo=list(t); s3={}
    keys=['74','77','79','80','81','82','85','87']
    for ki,grps in enumerate(SLOTS):
        for ji,(eq,_,grp) in enumerate(dispo):
            if grp in grps: s3[keys[ki]]=eq; dispo.pop(ji); break
        else:
            if dispo: s3[keys[ki]]=dispo.pop(0)[0]
    g=s3.get; fb=list(epg.values())[0]
    r32=[
        (d['A'],d['B']),(p['E'],g('74',fb)),(p['F'],d['C']),(p['C'],d['F']),
        (p['I'],g('77',fb)),(d['E'],d['I']),(p['A'],g('79',fb)),(p['L'],g('80',fb)),
        (p['D'],g('81',fb)),(p['G'],g('82',fb)),(d['K'],d['L']),(p['H'],d['J']),
        (p['B'],g('85',fb)),(p['J'],d['H']),(p['K'],g('87',fb)),(d['D'],d['G']),
    ]
    # ✅ Bracket officiel FIFA 2026
    v=[me(a,b) for a,b in r32]
    rda=[me(v[1],v[4]),me(v[0],v[2]),me(v[10],v[11]),me(v[8],v[9])]
    rat=[me(v[3],v[5]),me(v[6],v[7]),me(v[13],v[15]),me(v[12],v[14])]
    qd=[me(rda[0],rda[1]),me(rda[2],rda[3])]
    qa=[me(rat[0],rat[1]),me(rat[2],rat[3])]
    return me(me(qd[0],qd[1]),me(qa[0],qa[1]))

print(f'Lancement {N_SIMULATIONS} simulations...')
t1=time.time()
champions=[tmc() for _ in range(N_SIMULATIONS)]
print(f'✅ Terminé en {time.time()-t1:.1f}s')

c=Counter(champions)
print(f'\n{"="*50}\n  MONTE CARLO — {N_SIMULATIONS} SIMULATIONS\n{"="*50}')
for eq,n in c.most_common(15):
    pct=n/N_SIMULATIONS*100
    print(f'  {eq:<25} {n:>6}  {pct:>5.1f}%  {"█"*int(pct/1.5)}')
eq_mc,nb_mc=c.most_common(1)[0]
print(f'\n  🏆 {eq_mc} — {nb_mc/N_SIMULATIONS*100:.1f}%')

Précalcul 2256 paires...
✅ Cache en 25s
Lancement 5000 simulations...
✅ Terminé en 0.3s

  MONTE CARLO — 5000 SIMULATIONS
  France                       516   10.3%  ██████
  Spain                        418    8.4%  █████
  Argentina                    400    8.0%  █████
  England                      378    7.6%  █████
  Germany                      369    7.4%  ████
  Brazil                       331    6.6%  ████
  Netherlands                  330    6.6%  ████
  Portugal                     312    6.2%  ████
  Belgium                      210    4.2%  ██
  Japan                        174    3.5%  ██
  Croatia                      150    3.0%  ██
  Morocco                      138    2.8%  █
  Mexico                       137    2.7%  █
  Senegal                      119    2.4%  █
  United States                 98    2.0%  █

  🏆 France — 10.3%
